In [112]:
import os
import email
import email.policy
import numpy as np
from sklearn.model_selection import train_test_split
import re
from bs4 import BeautifulSoup
import nltk
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

In [24]:
# 1. Define The Paths To The Extracted Dataset Directories
HAM_DIR = "Data/easy_ham"
SPAM_DIR = "Data/spam"

In [25]:
# 2. Load The Lists Containing All Filenames In Each Directory
ham_filenames = os.listdir(HAM_DIR)
spam_filenames = os.listdir(SPAM_DIR)

In [26]:
# 3. Print The Total Number Of Email Files Found In Each Category
print("Total Ham emails:", len(ham_filenames))
print("Total Spam emails:", len(spam_filenames))


Total Ham emails: 2551
Total Spam emails: 503


In [27]:
# 4. # Define A Function To Load A Raw Email File
def load_email(directory, filename):
    path = os.path.join(directory, filename)
    with open(path, "rb") as f:
              return email.parser.BytesParser(policy=email.policy.default).parse(f)

In [28]:
# 5. Load One Sample Email From Each Category
sample_ham = load_email(HAM_DIR, ham_filenames[2])
sample_spam = load_email(SPAM_DIR, spam_filenames[2])

In [29]:
# Print The Subject Of The Spam Email
print("Spam Subject", sample_spam["Subject"])

# Print The First 500 Characters Of The Spam Content
print("Spam Content:/n", sample_spam.get_content().strip()[:500])

    

Spam Subject Life Insurance - Why Pay More?
Spam Content:/n <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN">
<HTML><HEAD>
<META content="text/html; charset=windows-1252" http-equiv=Content-Type>
<META content="MSHTML 5.00.2314.1000" name=GENERATOR></HEAD>
<BODY><!-- Inserted by Calypso -->
<TABLE border=0 cellPadding=0 cellSpacing=2 id=_CalyPrintHeader_ rules=none 
style="COLOR: black; DISPLAY: none" width="100%">
  <TBODY>
  <TR>
    <TD colSpan=3>
      <HR color=black noShade SIZE=1>
    </TD></TR></TD></TR>
  <TR>
    <TD colSpan=3>
   


In [19]:
# 6. Create Empty Lists For The Verified Email Filenames
valid_ham_filenames = []
valid_spam_filenames = []

In [21]:
# 7. Filter The Ham Directory For Valid Emails
for filename in ham_filenames:
    try:
        current_email = load_email(HAM_DIR, filename)
        # Check If The Email Has A Valid Subject Header
        if current_email["Subject"] is not None:
            valid_ham_filenames.append(filename)
    except Exception: 
         # Ignore Any Corrupted Files That Fail To Parse
        pass

# 8. Filter The Spam Directory For Valid Emails
for filename in ham_filenames:
    try:
        cureent_email = load_email(HAM_DIR, filename)
        # Check If The Email Has A Valid Subject Header
        if current_email["Subjecct"] is not None:
            valid_spam_filenames.append(filename)
    except Exception:
         # Ignore Any Corrupted Files That Fail To Parse
        pass
        
        
        

In [22]:

# 9. Replace The Original Lists With The Cleaned Versions
ham_filenames = valid_ham_filenames
spam_filenames = valid_spam_filenames

In [23]:
# 10. Print The Counts
print("Verified Ham Count:", len(ham_filenames))
print("Verified Spam Count:", len(spam_filenames))

Verified Ham Count: 5102
Verified Spam Count: 0


In [39]:
# 11. Reload The Original Filenames From The Directories First
ham_filenames = os.listdir(HAM_DIR)
spam_filenames = os.listdir(SPAM_DIR)

In [41]:
# 12. Filter Out Non-Email Files Using Name Validation
ham_filenames = [f for f in ham_filenames if"." in f and "cmds" not in f]
spam_filenames = [f for f in spam_filenames if"." in f and "cmds" not in f]

In [42]:
# 13. Print The Newly Cleaned Counts
print("Cleaned Ham Count:", len(ham_filenames))
print("Cleaned Spam Count:", len(spam_filenames))


Cleaned Ham Count: 2551
Cleaned Spam Count: 502


In [47]:
# 14. Split the datasets into a training set and a test

# Combine All Filenames Into One Single Input Array
X = np.array(ham_filenames + spam_filenames)

y = np.array([0] * len(ham_filenames) + [1] * len(spam_filenames))


In [48]:
# 15. Print The Shapes To Verify Everything Matches Perfectly
print ("X Shape:", X.shape)
print ("y Shape:", y.shape)

X Shape: (3053,)
y Shape: (3053,)


In [50]:
# 16. Split The Dataset Into Train And Test Sets Using Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [51]:
# 17. Print The Lengths To Verify The 80-20 Split
print("Train Set Size:", len(X_train))
print("Test Set Size:", len(X_test))
      

Train Set Size: 2442
Test Set Size: 611


In [64]:

# 18. Define A Function To Extract And Clean Text From An Email Object
def email_to_text(email_obj):
    # Iterate Through All Parts If The Email Is Multipart
    for part in email_obj.walk():
        content_type = part.get_content_type()
        
        # Look For Plain Text Or HTML Content Types
        if content_type in ["text/plain", "text/html"]:
            try:
                content = part.get_content()
                text = str(content)
                
                # If The Content Is HTML, Strip All HTML Tags And DOCTYPE Declarations
                if content_type == "text/html":
                    text = BeautifulSoup(text, "html.parser").get_text()
                    
                return text
            except Exception as e:
                   # Print Error Message If Extraction Fails
                print(f"Debug: Error Reading Content -> {e}")
                
                pass
    return ""


   
        
    

In [65]:
# 19. Print Check Sample Spam 300
print(email_to_text(sample_spam)[:300])
















Save up to 70% on Life Insurance.
Why Spend More Than You Have To?

Life Quote Savings










Ensuring your 
      family's financial security is very important. Life Quote Savings makes 
      buying life insurance simple and affordable. We Provide FREE Access to The 
      Very B


In [66]:
!pip install nltk


   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 4.1 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 3.5 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [regex]
   ---------------------------------------- 0/2 [regex]
   ---------------------------------------- 0/2 [regex]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ---------------

In [130]:
# 20. Download The Required NLTK Components For Stemming
nltk.download('punkt', quiet=True)
stemmer = nltk.PorterStemmer()

# 21. Create A Custom Transformer For Data Preparation 
class EmailToWordCountsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lowercase=True, remove_punctuation=True,
                 replace_urls=True, replace_numbers=True, stemming=True):
        self.lowercase = lowercase
        self.remove_punctuation = remove_punctuation
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.stemming = stemming
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X, y=None):
        X_transformed = []
        for email in X:
            # Since The Data Is Aiready Strings We Use It Directly
            text = str(email) or ""


            # Clean HTML Tags If Present In The Text

            if "<html>" in text.lower() or "<body" in text.lower():
                text = BeautifulSoup(text, "html.parser").get_text()
            
            # Convert Text To Lowercase
            if self.lowercase:
                text = text.lower()
                
            # Replace URLs With A Uniform Placeholder Token
            if self.replace_urls:
                text = re.sub(r'(https?://\S+|www\.\S+)', 'URL', text)
                
            # Replace All Numbers With A Uniform Placeholder Token
            if self.replace_numbers:
                text = re.sub(r'\d+', 'NUMBER', text)
                
            # Remove Punctuation And Keep Alphanumeric Words
            if self.remove_punctuation:
                text = re.sub(r'\W+', ' ', text)
                
            # Perform Word Stemming To Reduce Words To Their Base Root
            if self.stemming:
                words = text.split()
                stemmed_words = [stemmer.stem(word) for word in words]
                text = " ".join(stemmed_words)
                
            X_transformed.append(text)
            
        return X_transformed

# 22. Initialize The Vectorizer To Create Sparse Vectors
# Limit The Vocabulary Size To The 1000 Most Frequent Words
vocabulary_size = 1000 
vectorizer = CountVectorizer(max_features=vocabulary_size)

In [131]:
# 23. Initialize The Custom Transformer
transformer = EmailToWordCountsTransformer()

# 24. Transform The Raw Training Emails Into Cleaned Text Strings
X_train_clean = transformer.transform(X_train)

# 25. Fit The Vectorizer On Training Text And Convert To Sparse Vectors
X_train_vectors = vectorizer.fit_transform(X_train_clean)

# 26. Transform The Raw Testing Emails Using The Already Fitted Vectorizer
X_test_clean = transformer.transform(X_test)
X_test_vectors = vectorizer.transform(X_test_clean)

# 27. Print The Shape Of The Resulting Training Feature Matrix
print(f"Training Vectors Shape: {X_train_vectors.shape}")


Training Vectors Shape: (2442, 1000)


In [132]:
# 28. Initialize And Train The Logistic Regression Model
log_clf = LogisticRegression(solver="lbfgs", max_iter=1000, class_weight="balanced", random_state=42)
log_clf.fit(X_train_vectors, y_train)

# 29. Make Predictions On The Training Set To Declare The Variable
y_train_pred_log = log_clf.predict(X_train_vectors)

# 30. Print Metrics For Logistic Regression Classifier
print("Logistic Regression Train Performance")
print(f"Precision Score: {precision_score(y_train, y_train_pred_log):.2f}")
print(f"Recall Score: {recall_score(y_train, y_train_pred_log):.2f}\n")

# 31. Initialize And Train The Random Forest Classifier
forest_clf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
forest_clf.fit(X_train_vectors, y_train)

# 32. Make Predictions On The Training Set For Random Forest
y_train_pred_forest = forest_clf.predict(X_train_vectors)

# 33. Print Metrics For Random Forest Classifier
print("Random Forest Train Performance")
print(f"Precision Score: {precision_score(y_train, y_train_pred_forest):.2f}")
print(f"Recall Score: {recall_score(y_train, y_train_pred_forest):.2f}")

Logistic Regression Train Performance
Precision Score: 1.00
Recall Score: 0.39

Random Forest Train Performance
Precision Score: 0.25
Recall Score: 1.00


In [133]:
# 34. Count The Number Of Ham And Spam Samples In The Training Set
unique, counts = np.unique(y_train, return_counts=True)
print("Training Labels Distribution")
print(dict(zip(unique, counts)))

Training Labels Distribution
{np.int64(0): np.int64(2040), np.int64(1): np.int64(402)}


In [134]:
# 35. Get The Probability Predictions For The Training Set
# This Returns Two Columns: Probability Of Being Ham And Probability Of Being Spam
log_clf_probs = log_clf.predict_proba(X_train_vectors)[:, 1]

# 36. Set A Custom Threshold To Balance Precision And Recall
# Lowering It From 0.50 Makes The Model More Aggressive In Catching Spam

custom_threshold = 0.25
y_train_pred_custom = (log_clf_probs >= custom_threshold).astype(int)

# 37. Print The New Adjusted Metrics For Logistic Regression
print(f"Logistic Regression Performance With Threshold {custom_threshold}")
print(f"Adjusted Precision Score: {precision_score(y_train, y_train_pred_custom):.2f})")
print(f"Adjusted Recall Score: {recall_score(y_train, y_train_pred_custom):.2f})")

Logistic Regression Performance With Threshold 0.25
Adjusted Precision Score: 0.16)
Adjusted Recall Score: 1.00)


In [135]:
# 37. Initialize The TF IDF Vectorizer With A Larger Vocabulary
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# 38. Fit And Transform The Cleaned Training Text Content
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_clean)

# 39. Train A Fresh Logistic Regression Model On TF IDF Features
log_clf_tfidf = LogisticRegression(solver="lbfgs", max_iter=1000, class_weight="balanced", random_state=42)
log_clf_tfidf.fit(X_train_tfidf, y_train)

# 40. Make Predictions To See The Natural Baseline Performance
y_train_pred_tfidf = log_clf_tfidf.predict(X_train_tfidf)

# 41. Print The New Adjusted Metrics For Logistic Regression
print("TF-IDF Logistic Regression Performance")
print(f"Precision Score: {precision_score(y_train, y_train_pred_tfidf):.2f})")
print(f"Recall Score: {recall_score(y_train, y_train_pred_tfidf):.2f})")

TF-IDF Logistic Regression Performance
Precision Score: 1.00)
Recall Score: 1.00)


In [136]:
# 42. Transform The Cleaned Testing Text Using The Fitted TF IDF Vectorizer
X_test_tfidf = tfidf_vectorizer.transform(X_test_clean)

# 43. Make Predictions On The Unknown Test Set Data
y_test_pred_tfidf = log_clf_tfidf.predict(X_test_tfidf)

# 44. Print The New Adjusted Metrics For Logistic Regression
print("TF-IDF Logistic Regression Test Performance")
print(f"Final Test Precision Score: {precision_score(y_train, y_train_pred_tfidf):.2f})")
print(f" Final Test Recall Score: {recall_score(y_train, y_train_pred_tfidf):.2f})")


TF-IDF Logistic Regression Test Performance
Final Test Precision Score: 1.00)
 Final Test Recall Score: 1.00)


In [137]:
# 45. Save The Trained Logistic Regression Model To A File
joblib.dump(log_clf_tfidf, "spam_classifier_model.pkl")

# 46. Save The Fitted TF IDF Vectorizer To A File
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

# 47. Print A Success Message To Confirm The Files Were Created
print("Model And Vectorizer Saved Successfully")

Model And Vectorizer Saved Successfully


In [138]:
# 48. Feature Use - First Attempt

# 49. Load The Saved Model And Vectorizer Back Into Memory
loaded_model = joblib.load("spam_classifier_model.pkl")
loaded_vectorizer = joblib.load("tfidf_vectorizer.pkl")

# 50. Define A New Custom Email Text To Test The System
new_email = "Check out our guaranteed low prices on pharmaceuticals. Save money today on Viagra! Unsubscribing instructions inside."

# 51. Initialize Your Existing Transformer Object To Clean The New Email
transformer = EmailToWordCountsTransformer()

# 51. Clean The New Email Text Using The Pipeline Transformations
# We Put The Email Inside A List Because The Transformer Expects An Iterable
new_email_clean = transformer.transform([new_email])

# 52. Transform The Cleaned Text Content Into TF IDF Vectors
new_email_vector = loaded_vectorizer.transform([new_email])

# 52. Make A Prediction Using The Loaded Classifier Model
prediction = loaded_model.predict(new_email_vector)


# 53. Print The Final Prediction Result To The Screen
if prediction[0] == 1:
    print("Prediction Result: SPAM")
else:
    print("Prediction Result: HAM")

Prediction Result: HAM


In [139]:
# 54. Define A Strong Traditional Spam Email For Testing
test_email = "Check out our guaranteed low prices on pharmaceuticals. Save money today on Viagra! Unsubscribing instructions inside."

# 55. Use The Active Living Transformer From Memory To Clean Text
transformer = EmailToWordCountsTransformer()
test_email_clean = transformer.transform([test_email])

# 56. Use The Active Living Vectorizer From Memory To Transform Text
test_email_vector = tfidf_vectorizer.transform(test_email_clean)

# 57. Use The Active Living Model From Memory To Predict
live_prediction = log_clf_tfidf.predict(test_email_vector)


# 58. Print The Final Prediction Result To The Screen
if prediction[0] == 1:
    print("Prediction Result: SPAM")
else:
    print("Prediction Result: HAM")

Prediction Result: HAM


In [140]:

# 59. Find The Index Of The First Actual Spam Email In Your Dataset
# We Look Into y_train For The Value 1 Which Represents Spam
spam_indices = [i for i, label in enumerate(y_train) if label == 1]
first_spam_index = spam_indices[0]

# 60. Extract That Specific Real Spam Email String From X_train_clean
real_dataset_spam = [X_train_clean[first_spam_index]]

# 61. Transform That Real Dataset Spam Into TF IDF Vectors
real_spam_vector = tfidf_vectorizer.transform(real_dataset_spam)

# 62. Make A Prediction On This Verified Dataset Email
dataset_prediction = log_clf_tfidf.predict(real_spam_vector)

# 63. Print The Dataset Verification Result To The Screen
print(f"Testing With Real Dataset Spam At Index: {first_spam_index}")
if dataset_prediction[0] == 1:
    print("Prediction Result: SPAM")
else:
    print("Prediction Result: HAM")


Testing With Real Dataset Spam At Index: 6
Prediction Result: SPAM


In [143]:
# 65. Real Use Case Feature From 2000s - Second Attempt
# 66. Define A Long Traditional Spam Email With Multiple Keywords
long_custom_spam = """
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN">
<HTML><HEAD><META content="text/html; charset=windows-1252" http-equiv=Content-Type></HEAD>
<BODY bgcolor="#ffffff">
<TABLE border=0 cellPadding=0 cellSpacing=2 id=_CalyPri>
<TR><TD>
<FONT face="Arial" size="4" color="#ff0000"><B>FREE MORTGAGE INSURANCE QUOTE!</B></FONT><BR>
<FONT face="Arial" size=2>Dear Homeowner, You have been selected to win a guaranteed cash prize!<BR>
Save money today on pharmaceuticals and viagra online. High success rates guaranteed.<BR>
Click this link immediately: <A href="http://spam-link.com">URL</A> to claim your millions.<BR>
Unsubscribe instructions: To remove your email from this unsolicited commercial mailing list, click URL.</FONT>
</TD></TR>
</TABLE>
</BODY></HTML>
"""

# 67. Clean The Long Custom Email Using The Pipeline
long_spam_clean = transformer.transform([long_custom_spam])

# 68. Transform Into TF IDF Vectors
long_spam_vector = tfidf_vectorizer.transform(long_spam_clean)

# 69. Make A Prediction Using The Active Model
long_prediction = log_clf_tfidf.predict(long_spam_vector)

 #70. Print The Result To Verify The Length Theory
print("Testing With Long Custom Spam Emaiil")
if long_prediction == 1:
    print("Prediction Result: SPAM")
else:
    print("Prediction Result: HAM")


Testing With Long Custom Spam Emaiil
Prediction Result: HAM


In [145]:
# 71. Get The Raw Probability Values For The Long Custom Spam Email
# This Calculates The Exact Percentage Of How Likely It Is To Be Spam
long_spam_probs = log_clf_tfidf.predict_proba(long_spam_vector)[:, 1]

# 72. Set A Lower Custom Threshold Since Clean Short Input Has Lower Total Weights
custom_live_threshold = 0.10
final_live_prediction = (long_spam_probs >= custom_live_threshold).astype(int)[0]

# 78. Print The Raw Probability To Understand The Math Behind The Decision
print(f"Raw Math Probability For Being Spam: {long_spam_probs[0]:.4f}")

 #79. Print The Result To Verify The Length Theory
print(f"Tetsing With Adjusted Threshold:) {custom_live_threshold}")
if long_prediction == 1:
    print(" Adjusted Prediction Result: SPAM")
else:
    print("Adjusted Prediction Result: HAM")


Raw Math Probability For Being Spam: 0.3994
Tetsing With Adjusted Threshold:) 0.1
Adjusted Prediction Result: HAM


In [146]:
# Third Attempt - Real Feature Case Use From Emails Of 2000s With Different Metrics
# 80. Get The Raw Probability Vector For The Long Custom Spam Email
# The Column One Represents The Exact Probability Of Being Spam
long_spam_prob_value = log_clf_tfidf.predict_proba(long_spam_vector)[0, 1]

# 81. Set A Lower Custom Sensitive Threshold For Short Custom Texts
custom_live_threshold = 0.10

# 82. Print The Raw Probability To Understand The Math Behind The Decision
print(f"Raw Math Probability For Being Spam: {long_spam_prob_value:.4f}")
print(f"Tetsing With Adjusted Threshold:) {custom_live_threshold}")

if long_spam_prob_value >= custom_live_threshold:
    print(" Adjusted Prediction Result: SPAM")
else:
    print("Adjusted Prediction Result: HAM")


Raw Math Probability For Being Spam: 0.3994
Tetsing With Adjusted Threshold:) 0.1
 Adjusted Prediction Result: SPAM
